In [14]:
import pandas as pd

In [15]:
df = pd.read_csv('enterprise_cost_dataset.csv')
df.head()

,Transaction_ID,Vendor,Category,Amount,Date,Department
0,1,IBM,Consulting,13803,2024-01-01 00:00:00,IT
1,2,TCS,Software,50625,2024-01-01 01:00:00,Operations
2,3,Infosys,Cloud,94361,2024-01-01 02:00:00,Finance
3,4,IBM,Consulting,14698,2024-01-01 03:00:00,Operations
4,5,Google Cloud,Consulting,35560,2024-01-01 04:00:00,Operations


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║        AI SPEND INTELLIGENCE AGENT — Enterprise Cost Optimizer       ║
║        MVP System: Detection · Decision Engine · API · Reports       ║
╚══════════════════════════════════════════════════════════════════════╝

HOW TO RUN:
  As API server:   uvicorn ai_spend_intelligence_agent:app --reload
  In Jupyter:      Run cells section-by-section (see __main__ block)
  As script:       python ai_spend_intelligence_agent.py

DEPENDENCIES:
  pip install pandas scikit-learn fastapi uvicorn
"""

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# # ─────────────────────────────────────────────────────────────────────────────
# # SECTION 0 ── SAMPLE DATA GENERATOR
# # (Skip this section if you already have your own DataFrame)
# # ─────────────────────────────────────────────────────────────────────────────

# def generate_sample_data(n: int = 100, seed: int = 42) -> pd.DataFrame:
#     """
#     Creates a realistic-looking spend dataset with planted anomalies
#     so the detection engine has something interesting to find.
#     """
#     np.random.seed(seed)
#     rng = pd.date_range("2024-01-01", periods=n, freq="D")

#     vendors     = ["AWS", "Salesforce", "Oracle", "Zoom", "Slack",
#                    "Adobe", "Microsoft", "HubSpot", "Twilio", "Stripe"]
#     categories  = ["Cloud", "SaaS", "Infra", "Comms", "Analytics",
#                    "Marketing", "Security", "HR", "Finance", "Other"]
#     departments = ["Engineering", "Sales", "Marketing",
#                    "HR", "Finance", "Operations"]

#     df = pd.DataFrame({
#         "Transaction_ID": [f"TXN{1000 + i}" for i in range(n)],
#         "Vendor":         np.random.choice(vendors, n),
#         "Category":       np.random.choice(categories, n),
#         "Amount":         np.round(np.random.exponential(scale=5000, size=n), 2),
#         "Date":           np.random.choice(rng, n),
#         "Department":     np.random.choice(departments, n),
#     })

#     # ── Plant anomalies ──────────────────────────────────────────────────────
#     # 1. Duplicate payments (rows 5–7 mirror row 0)
#     for i in [5, 6, 7]:
#         df.loc[i, ["Vendor", "Amount", "Date"]] = df.loc[0, ["Vendor", "Amount", "Date"]]

#     # 2. High-spend outliers
#     df.loc[10, "Amount"] = 180_000
#     df.loc[20, "Amount"] = 220_000

#     return df.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 ── DATA PROCESSING
# ─────────────────────────────────────────────────────────────────────────────

class DataProcessor:
    """
    Loads, cleans, and prepares the spend DataFrame for analysis.
    Handles: type conversion, missing values, duplicates.
    """

    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # 1a. Normalize column names
        df.columns = [c.strip().replace(" ", "_") for c in df.columns]

        # 1b. Convert Date → datetime (coerce bad values to NaT)
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

        # 1c. Convert Amount → numeric (coerce bad values to NaN)
        df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

        # 1d. Drop rows where critical fields are missing
        before = len(df)
        df.dropna(subset=["Transaction_ID", "Amount", "Date"], inplace=True)
        dropped = before - len(df)
        if dropped:
            print(f"  [DataProcessor] Dropped {dropped} rows with missing critical fields.")

        # 1e. Fill other missing text fields with "Unknown"
        for col in ["Vendor", "Category", "Department"]:
            if col in df.columns:
                df[col].fillna("Unknown", inplace=True)

        # 1f. Remove identical full-row duplicates (keep first)
        df.drop_duplicates(inplace=True)
        df.reset_index(drop=True, inplace=True)

        print(f"  [DataProcessor] Clean dataset: {len(df)} rows ready for analysis.")
        return df


# ─────────────────────────────────────────────────────────────────────────100────
# SECTION 2 ── ANOMALY DETECTION
# ─────────────────────────────────────────────────────────────────────────────

class AnomalyDetector:
    """
    Two-layer detection:
      A. Rule-based  — deterministic, zero false-negative risk for known patterns
      B. ML-based    — Isolation Forest for unknown/novel anomalies
    """

    def __init__(self, use_ml: bool = True, contamination: float = 0.05):
        self.use_ml = use_ml
        self.contamination = contamination   # expected anomaly fraction

    # ── 2A. Rule-Based ───────────────────────────────────────────────────────

    def detect_duplicates(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Flags transactions sharing the same (Vendor, Amount, Date).
        The first occurrence is kept as 'original'; subsequent ones are duplicates.
        """
        key = ["Vendor", "Amount", "Date"]
        # Mark every row that has at least one twin
        mask = df.duplicated(subset=key, keep=False)
        # Among duplicates, flag all but the first as the actual duplicate payment
        dup_mask = df.duplicated(subset=key, keep="first") & mask

        dups = df[dup_mask].copy()
        dups["Issue_Type"] = "Duplicate Payment"
        return dups

    def detect_high_spend(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        
        df["threshold"] = df.groupby("Category")["Amount"].transform(
            lambda x: x.quantile(0.95)
        )
        
        high = df[df["Amount"] > df["threshold"]].copy()
        high["Issue_Type"] = "High Spend Anomaly"
        
        return high.drop(columns=["threshold"])

    # ── 2B. ML-Based (Isolation Forest) ─────────────────────────────────────

    def detect_ml_anomalies(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Uses sklearn's Isolation Forest on Amount alone (MVP scope).
        Returns rows the model labels as anomalies (-1).
        Extend the feature set (e.g., day-of-week, vendor frequency)
        for richer detection in production.
        """
        try:
            from sklearn.ensemble import IsolationForest
        except ImportError:
            print("  [ML] scikit-learn not found — skipping Isolation Forest.")
            return pd.DataFrame()

        X = df[["Amount"]].values
        model = IsolationForest(
            contamination=self.contamination,
            random_state=42,
            n_estimators=100,
        )
        preds = model.fit_predict(X)               # -1 = anomaly, +1 = normal

        ml_anomalies = df[preds == -1].copy()
        ml_anomalies["Issue_Type"] = "ML Anomaly (Isolation Forest)"
        return ml_anomalies

    # ── 2C. Orchestrator ─────────────────────────────────────────────────────

    def detect_all(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Runs all detectors and returns a de-duplicated anomaly DataFrame.
        A transaction may be flagged by multiple rules; we keep the most
        severe label (Duplicate > High Spend > ML Anomaly).
        """
        results = []

        # Rule-based
        dups  = self.detect_duplicates(df)
        highs = self.detect_high_spend(df)
        results.extend([dups, highs])

        # ML (optional)
        if self.use_ml:
            ml = self.detect_ml_anomalies(df)
            results.append(ml)

        if not results:
            return pd.DataFrame()

        combined = pd.concat(results, ignore_index=True)

        # De-duplicate: if the same Transaction_ID appears multiple times,
        # keep only the row with the highest-priority issue type.
        priority = {
            "Duplicate Payment": 1,
            "High Spend Anomaly": 2,
            "ML Anomaly (Isolation Forest)": 3,
        }
        combined["_priority"] = combined["Issue_Type"].map(priority)
        combined.sort_values("_priority", inplace=True)
        combined.drop_duplicates(subset=["Transaction_ID"], keep="first", inplace=True)
        combined.drop(columns=["_priority"], inplace=True)
        combined.reset_index(drop=True, inplace=True)

        print(f"  [AnomalyDetector] {len(dups)} duplicates | "
              f"{len(highs)} high-spend | "
              f"{len(combined) - len(dups) - len(highs)} ML-only anomalies detected.")
        return combined


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 ── DECISION ENGINE
# ─────────────────────────────────────────────────────────────────────────────

class DecisionEngine:
    """
    Maps each anomaly to a recommended action and quantifies savings.

    Business rules
    ──────────────
    Duplicate Payment      → Block Payment       → save 100 % of Amount
    High Spend Anomaly     → Investigate Vendor  → save 20 % of Amount (est.)
    ML Anomaly             → Review Transaction  → save 10 % of Amount (est.)
    Category Overspending  → Recommend Optim.    → save 15 % of Amount (est.)
    """

    ACTION_MAP = {
        "Duplicate Payment":               ("Block Payment",          1.00),
        "High Spend Anomaly":              ("Investigate Vendor",     0.20),
        "ML Anomaly (Isolation Forest)":   ("Review Transaction",     0.10),
        "Category Overspend":              ("Recommend Optimization", 0.15),
    }

    def apply(self, anomalies: pd.DataFrame, df_full: pd.DataFrame) -> pd.DataFrame:
        """
        Enriches the anomaly DataFrame with Action and Estimated_Savings columns.
        Also detects category-level overspending from the full dataset.
        """
        if anomalies.empty:
            print("  [DecisionEngine] No anomalies to process.")
            return pd.DataFrame(columns=[
                "Transaction_ID", "Vendor", "Category", "Amount",
                "Department", "Issue_Type", "Action", "Estimated_Savings"
            ])

        results = anomalies.copy()

        # ── 3a. Category overspend detection ────────────────────────────────
        # Flag the top-spending transaction per category that exceeds the
        # category mean by > 1.5 standard deviations.
        cat_stats = df_full.groupby("Category")["Amount"].agg(["mean", "std"])
        overspend_ids = []

        for cat, row in cat_stats.iterrows():
            cat_mean, cat_std = row["mean"], row["std"]
            if pd.isna(cat_std) or cat_std == 0:
                continue
            threshold = cat_mean + 1.5 * cat_std
            over = df_full[
                (df_full["Category"] == cat) &
                (df_full["Amount"] > threshold)
            ]
            overspend_ids.extend(over["Transaction_ID"].tolist())

        # Upgrade issue type for transactions not already flagged
        already_flagged = set(results["Transaction_ID"])
        new_overspends  = [
            df_full[df_full["Transaction_ID"] == tid].assign(
                Issue_Type="Category Overspend"
            )
            for tid in overspend_ids
            if tid not in already_flagged
        ]

        if new_overspends:
            results = pd.concat(
                [results] + new_overspends, ignore_index=True
            )

        # ── 3b. Map to actions + calculate savings ───────────────────────────
        def get_action(issue):
            return self.ACTION_MAP.get(issue, ("Manual Review", 0.05))[0]

        def get_savings(row):
            _, rate = self.ACTION_MAP.get(row["Issue_Type"], ("Manual Review", 0.05))
            return round(row["Amount"] * rate, 2)

        results["Action"]            = results["Issue_Type"].apply(get_action)
        results["Estimated_Savings"] = results.apply(get_savings, axis=1)

        # ── 3c. Select and order output columns ─────────────────────────────
        out_cols = [
            "Transaction_ID", "Vendor", "Category",
            "Amount", "Department", "Issue_Type",
            "Action", "Estimated_Savings"
        ]
        # Keep only columns that actually exist in results
        out_cols = [c for c in out_cols if c in results.columns]
        results = results[out_cols].reset_index(drop=True)

        print(f"  [DecisionEngine] {len(results)} issues processed. "
              f"Total estimated savings: ₹{results['Estimated_Savings'].sum():,.2f}")
        return results


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 ── IMPACT CALCULATOR
# ─────────────────────────────────────────────────────────────────────────────

class ImpactCalculator:
    """
    Aggregates the financial impact from the decision engine output.
    """

    def calculate(self, issues_df: pd.DataFrame) -> dict:
        if issues_df.empty:
            return {
                "total_issues":          0,
                "total_savings":         0.0,
                "by_issue_type":         {},
                "by_department":         {},
                "duplicate_count":       0,
                "high_spend_count":      0,
                "ml_anomaly_count":      0,
                "category_overspend_count": 0,
            }

        by_type = (
            issues_df.groupby("Issue_Type")["Estimated_Savings"]
            .agg(count="count", savings="sum")
            .round(2)
            .to_dict("index")
        )

        by_dept = {}
        if "Department" in issues_df.columns:
            by_dept = (
                issues_df.groupby("Department")["Estimated_Savings"]
                .sum()
                .round(2)
                .to_dict()
            )

        def _count(issue_type):
            return int((issues_df["Issue_Type"] == issue_type).sum())

        return {
            "total_issues":    len(issues_df),
            "total_savings":   round(issues_df["Estimated_Savings"].sum(), 2),
            "by_issue_type":   by_type,
            "by_department":   by_dept,
            "duplicate_count":          _count("Duplicate Payment"),
            "high_spend_count":         _count("High Spend Anomaly"),
            "ml_anomaly_count":         _count("ML Anomaly (Isolation Forest)"),
            "category_overspend_count": _count("Category Overspend"),
        }


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 ── REPORT GENERATOR
# ─────────────────────────────────────────────────────────────────────────────

class ReportGenerator:
    """
    Produces a human-readable narrative summary of the analysis.
    """

    def generate(self, impact: dict) -> str:
        if impact["total_issues"] == 0:
            return "✅ No anomalies detected. Your spend data looks clean!"

        lines = [
            "═" * 60,
            "  AI SPEND INTELLIGENCE — EXECUTIVE SUMMARY",
            "═" * 60,
            f"  Analysis timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            "  FINDINGS",
            f"  ▸ Total issues detected    : {impact['total_issues']}",
            f"  ▸ Duplicate payments        : {impact['duplicate_count']}",
            f"  ▸ High-spend anomalies      : {impact['high_spend_count']}",
            f"  ▸ ML-detected anomalies     : {impact['ml_anomaly_count']}",
            f"  ▸ Category overspends       : {impact['category_overspend_count']}",
            "",
            f"  💰 ESTIMATED RECOVERABLE SAVINGS : ₹{impact['total_savings']:>12,.2f}",
            "",
        ]

        if impact["by_issue_type"]:
            lines.append("  BREAKDOWN BY ISSUE TYPE")
            for issue, vals in impact["by_issue_type"].items():
                lines.append(
                    f"  ▸ {issue:<40} "
                    f"{vals['count']:>3} issues  |  ₹{vals['savings']:>10,.2f}"
                )
            lines.append("")

        if impact["by_department"]:
            lines.append("  BREAKDOWN BY DEPARTMENT")
            for dept, savings in sorted(
                impact["by_department"].items(), key=lambda x: -x[1]
            ):
                lines.append(f"  ▸ {dept:<30} ₹{savings:>10,.2f}")
            lines.append("")

        lines += [
            "  RECOMMENDED NEXT STEPS",
            "  1. Block all flagged duplicate payments immediately.",
            "  2. Escalate high-spend vendors to procurement for contract review.",
            "  3. Request ML-flagged transactions be reviewed by finance within 48 h.",
            "  4. Initiate category budget optimization with department heads.",
            "═" * 60,
        ]

        return "\n".join(lines)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 ── AGENT ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────


class SpendIntelligenceAgent:
    """
    Top-level orchestrator that wires all components together.
    Call agent.run(df) to get the full analysis in one shot.
    """

    def __init__(self, use_ml: bool = True):
        self.processor   = DataProcessor()
        self.detector    = AnomalyDetector(use_ml=use_ml)
        self.engine      = DecisionEngine()
        self.calculator  = ImpactCalculator()
        self.reporter    = ReportGenerator()

        # State (populated after run())
        self.clean_df    = None
        self.issues_df   = None
        self.impact      = None
        self.report      = None

    def run(self, df: pd.DataFrame) -> dict:
        print("\n🔍 Starting AI Spend Intelligence Analysis …\n")

        print("► [1/4] Processing data …")
        self.clean_df = self.processor.process(df)

        print("\n► [2/4] Detecting anomalies …")
        anomalies     = self.detector.detect_all(self.clean_df)

        print("\n► [3/4] Applying decision engine …")
        self.issues_df = self.engine.apply(anomalies, self.clean_df)

        print("\n► [4/4] Calculating impact …")
        self.impact   = self.calculator.calculate(self.issues_df)
        self.report   = self.reporter.generate(self.impact)

        print("\n" + self.report)

        return {
            "clean_data":  self.clean_df,
            "issues":      self.issues_df,
            "impact":      self.impact,
            "report":      self.report,
        }


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 ── FASTAPI
# ─────────────────────────────────────────────────────────────────────────────
# Run with:  uvicorn ai_spend_intelligence_agent:app --reload
# Swagger:   http://127.0.0.1:8000/docs
# ─────────────────────────────────────────────────────────────────────────────

try:
    from fastapi import FastAPI
    from fastapi.responses import JSONResponse
    import json

    app   = FastAPI(
        title="AI Spend Intelligence Agent",
        description="Enterprise Cost Optimization API",
        version="1.0.0",
    )

    # ── Shared state (initialized at startup) ────────────────────────────────
    _agent  = SpendIntelligenceAgent(use_ml=True)
    _result = None

    def _ensure_analysed():
        """Lazy-run the agent if it hasn't been called yet."""
        global _result
        if _result is None:
            raw_df   = generate_sample_data(n=100)
            _result  = _agent.run(raw_df)

    def _df_to_json(df: pd.DataFrame):
        """Convert DataFrame to JSON-serialisable list of dicts."""
        return json.loads(df.to_json(orient="records", date_format="iso"))

    # ── Endpoints ─────────────────────────────────────────────────────────────

    @app.get("/", tags=["Health"])
    def root():
        return {"status": "ok", "message": "AI Spend Intelligence Agent is running."}

    @app.get("/transactions", tags=["Data"])
    def get_transactions():
        """Return the full cleaned transaction dataset."""
        _ensure_analysed()
        return JSONResponse(content=_df_to_json(_agent.clean_df))

    @app.get("/anomalies", tags=["Detection"])
    def get_anomalies():
        """Return all detected anomalies with actions and savings estimates."""
        _ensure_analysed()
        return JSONResponse(content=_df_to_json(_agent.issues_df))

    @app.get("/impact", tags=["Impact"])
    def get_impact():
        """Return aggregated financial impact summary."""
        _ensure_analysed()
        return JSONResponse(content=_agent.impact)

    @app.get("/report", tags=["Report"])
    def get_report():
        """Return the human-readable executive summary."""
        _ensure_analysed()
        return {"report": _agent.report}

    @app.post("/analyse", tags=["Data"])
    def analyse(payload: dict):
        """
        POST a custom dataset as JSON and run a fresh analysis.

        Expected body:
        {
          "records": [
            {"Transaction_ID": "TXN1001", "Vendor": "AWS", ...},
            ...
          ]
        }
        """
        global _result
        try:
            df = pd.DataFrame(payload.get("records", []))
            _result = _agent.run(df)
            return {"status": "ok", "issues_found": _agent.impact["total_issues"]}
        except Exception as e:
            return JSONResponse(status_code=400, content={"error": str(e)})

except ImportError:
    print("  [API] FastAPI not installed — API endpoints disabled.")
    print("        Install with:  pip install fastapi uvicorn")
    app = None


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 ── MAIN (Optimized for your existing dataset)
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # 1. PRIORITY: Check for your existing 'df' variable
    if 'df' in locals():
        print("✅ Found existing dataset 'df'. Running analysis...")
        
        # Use a copy to protect your original notebook variable
        working_df = df.copy()
        
        # Initialize and run
        agent = SpendIntelligenceAgent(use_ml=True)
        result = agent.run(working_df)
        
        # Display results
        print("\n" + "="*60)
        print("🔍 TOP 10 LEAKAGE ANOMALIES")
        print("="*60)
        if not result["issues"].empty:
            print(result["issues"].head(10).to_string(index=False))
        else:
            print("No anomalies detected in your dataset.")


    # 3. OPTIONAL: Export results for reporting
    if result["issues"] is not None and not result["issues"].empty:
        result["issues"].to_csv("cost_leakage_report.csv", index=False)
        print("\n📂 Report exported to 'cost_leakage_report.csv'")

✅ Found existing dataset 'df'. Running analysis...

🔍 Starting AI Spend Intelligence Analysis …

► [1/4] Processing data …
  [DataProcessor] Clean dataset: 10000 rows ready for analysis.

► [2/4] Detecting anomalies …
  [AnomalyDetector] 0 duplicates | 502 high-spend | 222 ML-only anomalies detected.

► [3/4] Applying decision engine …
  [DecisionEngine] 724 issues processed. Total estimated savings: ₹25,506,829.40

► [4/4] Calculating impact …

════════════════════════════════════════════════════════════
  AI SPEND INTELLIGENCE — EXECUTIVE SUMMARY
════════════════════════════════════════════════════════════
  Analysis timestamp : 2026-03-28 17:53:23

  FINDINGS
  ▸ Total issues detected    : 724
  ▸ Duplicate payments        : 0
  ▸ High-spend anomalies      : 502
  ▸ ML-detected anomalies     : 222
  ▸ Category overspends       : 0

  💰 ESTIMATED RECOVERABLE SAVINGS : ₹25,506,829.40

  BREAKDOWN BY ISSUE TYPE
  ▸ High Spend Anomaly                       502 issues  |  ₹25,459,632.40


In [17]:
# Initialize agent
agent = SpendIntelligenceAgent(use_ml=True)

# Run on your dataset
result = agent.run(df)

# View results
result["issues"].head(50)


🔍 Starting AI Spend Intelligence Analysis …

► [1/4] Processing data …
  [DataProcessor] Clean dataset: 10000 rows ready for analysis.

► [2/4] Detecting anomalies …
  [AnomalyDetector] 0 duplicates | 502 high-spend | 222 ML-only anomalies detected.

► [3/4] Applying decision engine …
  [DecisionEngine] 724 issues processed. Total estimated savings: ₹25,506,829.40

► [4/4] Calculating impact …

════════════════════════════════════════════════════════════
  AI SPEND INTELLIGENCE — EXECUTIVE SUMMARY
════════════════════════════════════════════════════════════
  Analysis timestamp : 2026-03-28 17:53:23

  FINDINGS
  ▸ Total issues detected    : 724
  ▸ Duplicate payments        : 0
  ▸ High-spend anomalies      : 502
  ▸ ML-detected anomalies     : 222
  ▸ Category overspends       : 0

  💰 ESTIMATED RECOVERABLE SAVINGS : ₹25,506,829.40

  BREAKDOWN BY ISSUE TYPE
  ▸ High Spend Anomaly                       502 issues  |  ₹25,459,632.40
  ▸ ML Anomaly (Isolation Forest)            222 is

,Transaction_ID,Vendor,Category,Amount,Department,Issue_Type,Action,Estimated_Savings
0,386,Wipro,Infrastructure,99833,IT,High Spend Anomaly,Investigate Vendor,19966.6
1,379,TCS,Consulting,417006,IT,High Spend Anomaly,Investigate Vendor,83401.2
2,377,AWS,Consulting,669564,HR,High Spend Anomaly,Investigate Vendor,133912.8
3,358,AWS,Software,533239,Operations,High Spend Anomaly,Investigate Vendor,106647.8
4,308,TCS,Cloud,99923,HR,High Spend Anomaly,Investigate Vendor,19984.6
5,277,IBM,Infrastructure,99475,HR,High Spend Anomaly,Investigate Vendor,19895.0
6,250,Wipro,Software,365190,Finance,High Spend Anomaly,Investigate Vendor,73038.0
7,245,Wipro,Consulting,325185,IT,High Spend Anomaly,Investigate Vendor,65037.0
8,219,AWS,Infrastructure,98520,Operations,High Spend Anomaly,Investigate Vendor,19704.0
9,179,AWS,Cloud,99261,IT,High Spend Anomaly,Investigate Vendor,19852.2


In [18]:
df["Date"] = pd.to_datetime(df["Date"])

daily_spend = df.groupby("Date")["Amount"].sum().reset_index()

from sklearn.linear_model import LinearRegression
import numpy as np

daily_spend["day_num"] = np.arange(len(daily_spend))

X = daily_spend[["day_num"]]
y = daily_spend["Amount"]

model = LinearRegression()
model.fit(X, y)

future_days = 30

future_X = np.arange(len(daily_spend), len(daily_spend) + future_days).reshape(-1, 1)

predictions = model.predict(future_X)
predictions

array([60085.01934707, 60084.93968223, 60084.86001739, 60084.78035254,
       60084.7006877 , 60084.62102285, 60084.54135801, 60084.46169317,
       60084.38202832, 60084.30236348, 60084.22269863, 60084.14303379,
       60084.06336895, 60083.9837041 , 60083.90403926, 60083.82437441,
       60083.74470957, 60083.66504472, 60083.58537988, 60083.50571504,
       60083.42605019, 60083.34638535, 60083.2667205 , 60083.18705566,
       60083.10739082, 60083.02772597, 60082.94806113, 60082.86839628,
       60082.78873144, 60082.7090666 ])

In [19]:
df["is_anomaly"] = (df["Amount"] > df["Amount"].quantile(0.95)).astype(int)

In [20]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Use more features so the AI can find deeper patterns instead of just reading the Amount threshold
X = pd.get_dummies(df[["Amount", "Vendor", "Department", "Category"]])
y = df["is_anomaly"]

# Prevent perfect memorization to calculate a continuous reality-based risk probability
model = RandomForestClassifier(min_samples_split=10, random_state=42)
model.fit(X, y)

df["risk_score"] = model.predict_proba(X)[:, 1]


df.head(50)

,Transaction_ID,Vendor,Category,Amount,Date,Department,is_anomaly,risk_score
0,1,IBM,Consulting,13803,2024-01-01 00:00:00,IT,0,0.0
1,2,TCS,Software,50625,2024-01-01 01:00:00,Operations,0,0.0
2,3,Infosys,Cloud,94361,2024-01-01 02:00:00,Finance,0,0.0
3,4,IBM,Consulting,14698,2024-01-01 03:00:00,Operations,0,0.0
4,5,Google Cloud,Consulting,35560,2024-01-01 04:00:00,Operations,0,0.0
5,6,Infosys,Software,16376,2024-01-01 05:00:00,Finance,0,0.0
6,7,Infosys,Software,84654,2024-01-01 06:00:00,Marketing,0,0.0
7,8,IBM,Software,14117,2024-01-01 07:00:00,HR,0,0.0
8,9,Azure,Software,57381,2024-01-01 08:00:00,HR,0,0.0
9,10,Google Cloud,Software,38956,2024-01-01 09:00:00,Finance,0,0.0
